# Самостоятельная работа: задачи 13–15
По рабочей тетради 1–2. Цапков М.А. ЭПМО-01-26

## 13. Сравнение линейного и бинарного поиска

**Условие.** Отсортированные массивы размером 1 000, 10 000, 100 000 и 1 000 000. Искомое значение — последний элемент.

### 1. Алгоритм и структуры данных

Для каждого размера создаём список `0, 1, …, N−1`. Линейный поиск последовательно проверяет элементы. Бинарный проверяет середину и сужает область вдвое. Оба алгоритма возвращают индекс и счётчик проверенных элементов.

### 2. Обоснование выбора

Линейный поиск служит простым исходным вариантом. Бинарный применим, поскольку список отсортирован и поддерживает доступ по индексу. Последний элемент позволяет увидеть худший случай линейного поиска. Как в оценках тетради, считаем один проверенный элемент одним шагом сравнения; отдельные операторы `==`, `<` и проверки цикла дополнительно не считаем.

### 3. Вычислительная сложность и масштабирование

Линейный поиск — `O(N)`, бинарный — `O(log N)` в худшем случае. Дополнительная память обоих — `O(1)`. При увеличении N в 10 раз линейный поиск делает в 10 раз больше сравнений, а бинарный — примерно на 3–4 шага больше.

### 4. Программная реализация

In [1]:
def linear_search_count(values, target):
    count = 0
    for i in range(len(values)):
        count += 1
        if values[i] == target:
            return i, count
    return -1, count


def binary_search_count(values, target):
    left = 0
    right = len(values) - 1
    count = 0
    while left <= right:
        middle = (left + right) // 2
        count += 1
        if values[middle] == target:
            return middle, count
        if values[middle] < target:
            left = middle + 1
        else:
            right = middle - 1
    return -1, count


### 5. Проверка и результаты

Для каждого N проверяем, что оба алгоритма вернули индекс `N−1`. Дополнительно проверяем первый элемент, отсутствующее значение и пустой список. Результаты эксперимента:

Проверки `assert` сравнивают фактические ответы с ожидаемыми. Результаты выполнения приведены под ячейкой.


In [2]:
print("N        Линейный  Бинарный")
for n in [1_000, 10_000, 100_000, 1_000_000]:
    values = list(range(n))
    linear_index, linear_count = linear_search_count(values, n - 1)
    binary_index, binary_count = binary_search_count(values, n - 1)
    assert linear_index == binary_index == n - 1
    print(f"{n:<9}{linear_count:<10}{binary_count}")

# Проверяем также первый элемент, отсутствующий элемент и пустой список.
for search in [linear_search_count, binary_search_count]:
    assert search([10, 20, 30], 10)[0] == 0
    assert search([10, 20, 30], 25)[0] == -1
    assert search([], 10) == (-1, 0)
del values

N        Линейный  Бинарный
1000     1000      10
10000    10000     14
100000   100000    17
1000000  1000000   20


### 6.Вывод и возможная оптимизация

Получены числа сравнений: линейный — 1 000, 10 000, 100 000, 1 000 000; бинарный — 10, 14, 17, 20. Это соответствует `O(N)` и `O(log N)`. Для многократного поиска в большом отсортированном массиве предпочтителен бинарный поиск. Если исходные данные не отсортированы, при множестве запросов можно один раз отсортировать их, как в примере 8 тетради.


## 14. Проверка уникальности идентификаторов

**Условие.** 1 млн уникальных ID. Нужно обнаруживать повторы и добавлять новые ID после проверки отсутствия.

### 1. Алгоритм и структуры данных

Реализуем два варианта хранения: список и хеш-множество `set`. Для каждого входного ID проверяем наличие. Если он найден, записываем `True` — повтор. Иначе записываем `False` и добавляем ID. Следующие события проверяем уже с учётом добавлений.

### 2. Обоснование выбора

В списке оператор `in` выполняет линейный поиск. Хеш-множество подходит для проверки наличия, поскольку порядок ID не требуется. Этот выбор соответствует примеру 7 тетради.

### 3. Вычислительная сложность и масштабирование

Список: поиск `O(N)`, добавление в конец амортизированно `O(1)`. Для Q новых уникальных ID список растёт: суммарно `O(QN + Q²)`; при небольшом Q относительно N обычно используют оценку `O(QN)`. Множество: проверка и добавление в среднем `O(1)`, обработка потока — в среднем `O(Q)`, вместе с созданием множества — `O(N + Q)`. Хранение — `O(N + Q)`. При коллизиях худший поиск в множестве может быть `O(N)`.

### 4. Программная реализация

In [3]:
def check_list(ids, incoming):
    repeated = []
    for identifier in incoming:
        if identifier in ids:
            repeated.append(True)
        else:
            repeated.append(False)
            ids.append(identifier)
    return repeated


def check_set(ids, incoming):
    repeated = []
    for identifier in incoming:
        if identifier in ids:
            repeated.append(True)
        else:
            repeated.append(False)
            ids.add(identifier)
    return repeated


### 5. Проверка и результаты

Создаём 1 млн исходных ID: от 0 до 999 999. Обрабатываем `[0, 999999, 1000000, 1000000, 1000001]`. Ожидаемые признаки повторов: `[True, True, False, True, False]`. Итоговое количество ID — 1 000 002. Сравниваем ответы и содержимое обеих структур.

Проверки `assert` сравнивают фактические ответы с ожидаемыми. Результаты выполнения приведены под ячейкой.


In [4]:
n = 1_000_000
ids_list = list(range(n))
ids_set = set(ids_list)
incoming = [0, n - 1, n, n, n + 1]
list_result = check_list(ids_list, incoming)
set_result = check_set(ids_set, incoming)
print("Входные ID:", incoming)
print("Повторы (список):", list_result)
print("Повторы (множество):", set_result)
print("ID после обработки:", len(ids_set))
assert list_result == set_result == [True, True, False, True, False]
assert set(ids_list) == ids_set
assert len(ids_set) == n + 2
del ids_list, ids_set

Входные ID: [0, 999999, 1000000, 1000000, 1000001]
Повторы (список): [True, True, False, True, False]
Повторы (множество): [True, True, False, True, False]
ID после обработки: 1000002


### 6.* Вывод и возможная оптимизация

Для высокочастотной проверки выбираем хеш-множество: оно даёт среднее `O(1)` на событие. Ограничения — дополнительная память, отсутствие сортировки и возможные коллизии. Если вместе с ID нужно хранить запись, множество можно заменить словарём.


## 15. Поиск транзакций

**Условие.** До 10 млн транзакций `(ID, время, сумма)` и до 50 тыс. точечных запросов/с. Нужны временные диапазоны и добавление.

### 1. Алгоритм и структуры данных

Используем словарь `ID → транзакция` и список транзакций, отсортированный по времени. Каждая транзакция — кортеж `(время, ID, сумма в копейках)`.

1. По ID обращаемся к словарю.
2. Для диапазона бинарным поиском находим начало, затем собираем записи до конца диапазона включительно.
3. При добавлении проверяем уникальность ID и вставляем транзакцию в обе структуры.

Для поиска по времени используем бинарный поиск первой подходящей позиции и последовательный сбор записей диапазона.

### 2. Обоснование выбора

Один линейный поиск может потребовать `10 000 000 × 50 000 = 5 × 10¹¹` сравнений/с. Словарь подходит для частых точных запросов, а отсортированный список — для диапазонов. Совмещение двух структур позволяет обслуживать оба типа поиска.

### 3. Вычислительная сложность и масштабирование

Поиск ID — в среднем `O(1)`, диапазон — `O(log N + K)`, добавление — `O(N)` из-за вставки в список. Память — `O(N)`. С ростом N точный и диапазонный поиск остаются эффективными, но стоимость вставок растёт линейно.

### 4. Программная реализация

Язык — Python 3.


In [5]:
# Событие: (время, идентификатор).
# Бинарный поиск первой позиции с временем >= time.
def first_position(events, time):
    left = 0
    right = len(events)
    while left < right:
        middle = (left + right) // 2
        if events[middle][0] < time:
            left = middle + 1
        else:
            right = middle
    return left


def events_in_range(events, start, end):
    result = []
    i = first_position(events, start)
    while i < len(events) and events[i][0] <= end:
        result.append(events[i])
        i += 1
    return result


def add_transaction(by_id, by_time, identifier, time, amount):
    if identifier in by_id:
        return False
    transaction = (time, identifier, amount)
    position = first_position(by_time, time)
    by_time.insert(position, transaction)
    by_id[identifier] = transaction
    return True


### 5. Проверка и результаты

Добавляем четыре транзакции, в том числе две с одинаковым временем. Проверяем ID 3: ожидаем `(20, 3, 5000)`, для отсутствующего ID — `None`. В диапазоне `[10, 20]` должны находиться ID 2, 3, 4. Повтор ID 3 отклоняется, размер обеих структур остаётся равным 4. Также проверяем диапазон без результатов.

Проверки `assert` сравнивают фактические ответы с ожидаемыми. Результаты выполнения приведены под ячейкой.


In [6]:
by_id = {}
by_time = []
add_transaction(by_id, by_time, 1, 30, 15000)
add_transaction(by_id, by_time, 2, 10, 20000)
add_transaction(by_id, by_time, 3, 20, 5000)
add_transaction(by_id, by_time, 4, 20, 7000)
print("Транзакция с ID 3:", by_id.get(3))
print("Несуществующий ID:", by_id.get(99))
result = events_in_range(by_time, 10, 20)
print("Транзакции за время [10, 20]:", result)
assert by_id[3] == (20, 3, 5000)
assert {t[1] for t in result} == {2, 3, 4}
assert not add_transaction(by_id, by_time, 3, 40, 9000)
assert len(by_id) == len(by_time) == 4
assert events_in_range(by_time, 31, 40) == []

Транзакция с ID 3: (20, 3, 5000)
Несуществующий ID: None
Транзакции за время [10, 20]: [(10, 2, 20000), (20, 4, 7000), (20, 3, 5000)]


### 6.* Вывод и возможная оптимизация

Комбинация словаря и бинарного поиска эффективнее полного просмотра для разных видов запросов. При частых вставках список следует заменить сбалансированным деревом: добавление станет `O(log N)`. Для высокой нагрузки нужно учитывать память и согласованное обновление индексов. Небольшие тесты подтверждают корректность, но реальную пропускную способность нужно измерять отдельно.
